# Gold de geração distribuída — V-Energy

Consolida a geração distribuída (GD) integrando a Silver de GD com o cadastro de Unidades Consumidoras e o histórico mensal de leituras. A transformação inteira está neste notebook (sem `%run` e sem módulo `.py` externo). Produz:

1. `gd_uc`: tabela de contrato do portal (`id_uc`, `potencia_kw`, `data_conexao`).
2. `tb_gd_uc`: dimensão enriquecida por UC com GD (`gd_uc_sk`, recortes cadastrais, consumo médio e mediano antes/após a conexão, queda percentual e `flag_queda_acima_30_pct`).
3. `tb_gd_consumo_mensal`: fato no grão `UC × competência` com `flag_gd_ativa_competencia` (`competencia >= competencia_conexao`), `volume_analise_kwh` e `consumo_kwh_analitico`.
4. `tb_gd_resumo_mensal`: agregado mensal por `competência × município × bairro × classe × modalidade`. Conexões novas e potência ativa vêm do cadastro de GD cruzado com o calendário mensal completo, então a soma de `quantidade_novas_conexoes_mes` fecha com o total de UCs com GD mesmo quando falta leitura no mês da conexão.

Regras e dicionário de dados: [`docs/gold/geracao_distribuida.md`](../../docs/gold/geracao_distribuida.md).

O padrão `modo=validar` não grava nada: **Run all** valida as fontes, monta e confere as quatro tabelas em memória. Para gravar em `v-energy.gold`, o Job passa `modo=publicar` (ou troque o widget antes de executar), como na Gold de faturas. A exportação para o banco do portal é um passo separado, que junta num pacote único as tabelas do contrato de todas as Golds.


In [0]:
"""Transformações e validações da Gold de geração distribuída."""

from pyspark.sql import DataFrame, Window
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType, LongType

VERSAO_REGRAS = 'GOLD_GD_V1'
PADRAO_ID_UC = r'^UC\d{6}$'
PADRAO_MES = r'^[1-9][0-9]{3}-(0[1-9]|1[0-2])$'
MODALIDADES_VALIDAS = ['PROPRIA_UC', 'AUTOCONSUMO_REMOTO', 'COMPARTILHADA']
TIPOS_LEITURA_VALIDOS = ['REAL', 'ESTIMADA']
DIMENSOES_UC = ['municipio', 'bairro', 'classe']
FLAGS_LEITURAS_BLOQUEIO = [
    'flag_troca_medidor',
    'flag_sem_leitura_anterior',
    'flag_sinal_nao_confirmado',
]
COLUNAS_GD = [
    'id_uc',
    'potencia_kwp',
    'fl_correcao_escala',
    'energia_injetada_kwh_media_mensal',
    'kwh_por_kwp_mes',
    'data_conexao',
    'fl_data_inconsistente',
    'modalidade',
    'modalidade_origem',
    'fl_deduplicado',
]
COLUNAS_CADASTRO = [
    'id_uc',
    'municipio',
    'bairro',
    'classe',
    'subgrupo_tarifario',
    'alimentador',
    'data_ligacao',
    'status',
]
COLUNAS_LEITURAS = [
    'id_uc',
    'mes_referencia',
    'consumo_kwh',
    'tipo_leitura',
    'flag_sinal_corrigido',
    *FLAGS_LEITURAS_BLOQUEIO,
]


def _exigir_colunas(df: DataFrame, colunas: list[str], nome: str) -> None:
    ausentes = set(colunas) - set(df.columns)
    if ausentes:
        raise ValueError(f'{nome}: colunas ausentes: {sorted(ausentes)}')


def _validar_regras(df: DataFrame, regras: dict, nome: str) -> None:
    contagens = (
        df.agg(*[F.count(F.when(condicao, 1)).alias(regra) for regra, condicao in regras.items()])
        .first()
        .asDict()
    )
    falhas = {regra: n for regra, n in contagens.items() if n}
    if falhas:
        raise ValueError(f'{nome}: {falhas}')


def _exigir_chave(df: DataFrame, colunas: list[str], nome: str) -> None:
    if df.groupBy(*colunas).count().filter('count > 1').limit(1).count():
        raise ValueError(f'{nome}: chave duplicada {colunas}')


def _soma_decimal(coluna, escala: int = 2):
    return F.coalesce(F.sum(coluna), F.lit(0)).cast(DecimalType(28, escala))


def validar_fontes(gd_silver: DataFrame, cadastro_silver: DataFrame, leituras_silver: DataFrame):
    """Valida contrato estrutural, tipos, chaves e domínios das três Silvers."""
    _exigir_colunas(gd_silver, COLUNAS_GD, 'gd_silver')
    _exigir_colunas(cadastro_silver, COLUNAS_CADASTRO, 'cadastro_silver')
    _exigir_colunas(leituras_silver, COLUNAS_LEITURAS, 'leituras_silver')

    tipos_gd = {
        'id_uc': 'string',
        'potencia_kwp': 'decimal(10,2)',
        'fl_correcao_escala': 'boolean',
        'energia_injetada_kwh_media_mensal': 'decimal(10,2)',
        'kwh_por_kwp_mes': 'decimal(10,2)',
        'data_conexao': 'date',
        'fl_data_inconsistente': 'boolean',
        'modalidade': 'string',
        'modalidade_origem': 'string',
        'fl_deduplicado': 'boolean',
    }
    for col_name, esperado in tipos_gd.items():
        obtido = gd_silver.schema[col_name].dataType.simpleString()
        if obtido != esperado:
            raise ValueError(f'gd_silver: {col_name} precisa ser {esperado} (obtido: {obtido})')

    for col_name in ['id_uc', 'municipio', 'bairro', 'classe', 'subgrupo_tarifario', 'alimentador']:
        if cadastro_silver.schema[col_name].dataType.simpleString() != 'string':
            raise ValueError(f'cadastro_silver: {col_name} precisa ser string')
    if cadastro_silver.schema['data_ligacao'].dataType.simpleString() != 'date':
        raise ValueError('cadastro_silver: data_ligacao precisa ser date')

    if leituras_silver.schema['consumo_kwh'].dataType.typeName() not in {
        'integer',
        'long',
        'short',
        'decimal',
    }:
        raise ValueError('leituras_silver: consumo_kwh precisa ser inteiro ou decimal')
    for flag in ['flag_sinal_corrigido', *FLAGS_LEITURAS_BLOQUEIO]:
        if leituras_silver.schema[flag].dataType.simpleString() != 'boolean':
            raise ValueError(f'leituras_silver: {flag} precisa ser boolean')

    if not gd_silver.limit(1).count():
        raise ValueError('gd_silver: fonte vazia')
    if not cadastro_silver.limit(1).count():
        raise ValueError('cadastro_silver: fonte vazia')
    if not leituras_silver.limit(1).count():
        raise ValueError('leituras_silver: fonte vazia')

    _exigir_chave(gd_silver, ['id_uc'], 'gd_silver')
    _exigir_chave(cadastro_silver, ['id_uc'], 'cadastro_silver')
    _exigir_chave(leituras_silver, ['id_uc', 'mes_referencia'], 'leituras_silver')

    _validar_regras(
        gd_silver,
        {
            'id_uc_invalido': F.col('id_uc').isNull() | ~F.col('id_uc').rlike(PADRAO_ID_UC),
            'potencia_invalida': F.col('potencia_kwp').isNull() | (F.col('potencia_kwp') <= 0),
            'energia_invalida': (
                F.col('energia_injetada_kwh_media_mensal').isNull()
                | (F.col('energia_injetada_kwh_media_mensal') <= 0)
            ),
            'rendimento_invalido': (
                F.col('kwh_por_kwp_mes').isNull() | (F.col('kwh_por_kwp_mes') <= 0)
            ),
            'data_conexao_nula': F.col('data_conexao').isNull(),
            'modalidade_invalida': (
                F.col('modalidade').isNull() | ~F.col('modalidade').isin(MODALIDADES_VALIDAS)
            ),
            'flags_nulas': (
                F.col('fl_correcao_escala').isNull()
                | F.col('fl_data_inconsistente').isNull()
                | F.col('fl_deduplicado').isNull()
            ),
        },
        'gd_silver',
    )

    _validar_regras(
        cadastro_silver,
        {
            f'{c}_ausente': F.col(c).isNull() | (F.trim(F.col(c)) == '')
            for c in ['id_uc', *DIMENSOES_UC, 'subgrupo_tarifario', 'alimentador', 'status']
        },
        'cadastro_silver',
    )

    _validar_regras(
        leituras_silver,
        {
            'chave_nula': F.col('id_uc').isNull() | F.col('mes_referencia').isNull(),
            'competencia_invalida': ~F.col('mes_referencia').rlike(PADRAO_MES),
            'consumo_nulo': F.col('consumo_kwh').isNull(),
            'tipo_leitura_invalido': (
                F.col('tipo_leitura').isNull() | ~F.col('tipo_leitura').isin(TIPOS_LEITURA_VALIDOS)
            ),
        },
        'leituras_silver',
    )

    orfas_cadastro = (
        gd_silver.select('id_uc')
        .join(cadastro_silver.select('id_uc'), 'id_uc', 'left_anti')
        .limit(1)
        .count()
    )
    if orfas_cadastro:
        raise ValueError('gd_silver: há UCs ausentes em cadastro_silver')


def construir_gold_gd(
    gd_silver: DataFrame,
    cadastro_silver: DataFrame,
    leituras_silver: DataFrame,
) -> dict[str, DataFrame]:
    """Constrói as 4 tabelas Gold de geração distribuída a partir das 3 Silvers."""
    validar_fontes(gd_silver, cadastro_silver, leituras_silver)

    # 1. Tabela de contrato do portal: gd_uc (1 linha por UC com GD)
    df_gd_uc = gd_silver.select(
        F.col('id_uc'),
        F.col('potencia_kwp').cast(DecimalType(18, 2)).alias('potencia_kw'),
        F.col('data_conexao'),
    ).orderBy('id_uc')

    # 2. Contexto cadastral (_ctx) e Lookup (_lk) seguindo taxonomia Star Schema
    df_cadastro_ctx = cadastro_silver.select(
        'id_uc',
        'municipio',
        'bairro',
        'classe',
        'subgrupo_tarifario',
        'alimentador',
        'data_ligacao',
        'status',
    )
    w_sk = Window.orderBy('id_uc')
    df_gd_base = (
        gd_silver.join(df_cadastro_ctx, 'id_uc', 'inner')
        .withColumn('gd_uc_sk', F.row_number().over(w_sk).cast(LongType()))
        .withColumn('competencia_conexao', F.trunc(F.col('data_conexao'), 'month'))
        .withColumn(
            'faixa_potencia',
            F.when(F.col('potencia_kwp') <= F.lit(5), F.lit('ATE_5_KWP'))
            .when(F.col('potencia_kwp') <= F.lit(10), F.lit('DE_5_A_10_KWP'))
            .when(F.col('potencia_kwp') <= F.lit(25), F.lit('DE_10_A_25_KWP'))
            .otherwise(F.lit('ACIMA_25_KWP')),
        )
    )
    df_gd_lk = df_gd_base.select(
        'gd_uc_sk',
        'id_uc',
        'municipio',
        'bairro',
        'classe',
        'subgrupo_tarifario',
        'alimentador',
        'modalidade',
        'faixa_potencia',
        'potencia_kwp',
        'energia_injetada_kwh_media_mensal',
        'data_conexao',
        'competencia_conexao',
    )

    # 3. Leituras das UCs com GD: consumo original vs consumo analítico
    leitura_bloqueada = F.lit(False)
    for flag in FLAGS_LEITURAS_BLOQUEIO:
        leitura_bloqueada = leitura_bloqueada | F.coalesce(F.col(flag), F.lit(True))

    volume_utilizavel = F.when(
        ~leitura_bloqueada & (F.col('consumo_kwh') >= 0),
        F.col('consumo_kwh').cast(DecimalType(18, 2)),
    )
    situacao_leitura = (
        F.when(F.col('flag_troca_medidor'), F.lit('TROCA_MEDIDOR'))
        .when(F.col('flag_sem_leitura_anterior'), F.lit('SEM_LEITURA_ANTERIOR'))
        .when(F.col('flag_sinal_nao_confirmado'), F.lit('SINAL_NAO_CONFIRMADO'))
        .when(F.col('consumo_kwh') < 0, F.lit('NEGATIVO_NAO_CLASSIFICADO'))
        .when(F.col('consumo_kwh') == 0, F.lit('ZERO_KWH'))
        .otherwise(F.lit('VOLUME_POSITIVO'))
    )

    df_leituras_gd = (
        leituras_silver.join(df_gd_lk, 'id_uc', 'inner')
        .withColumn('competencia', F.to_date(F.concat(F.col('mes_referencia'), F.lit('-01'))))
        .withColumn(
            'flag_gd_ativa_competencia',
            F.col('competencia') >= F.col('competencia_conexao'),
        )
        .withColumn(
            'fase_gd',
            F.when(F.col('flag_gd_ativa_competencia'), F.lit('APOS_CONEXAO')).otherwise(
                F.lit('ANTES_CONEXAO')
            ),
        )
        .withColumn('consumo_kwh_silver', F.col('consumo_kwh').cast(DecimalType(18, 2)))
        .withColumn('volume_analise_kwh', volume_utilizavel)
        .withColumn('situacao_volume', situacao_leitura)
    )

    # Mediana histórica por UC (por fase e global) para imputação analítica de meses bloqueados
    w_uc_fase = Window.partitionBy('id_uc', 'fase_gd')
    w_uc_global = Window.partitionBy('id_uc')
    mediana_fase = F.percentile_approx(F.col('volume_analise_kwh'), 0.5).over(w_uc_fase)
    mediana_global = F.percentile_approx(F.col('volume_analise_kwh'), 0.5).over(w_uc_global)

    df_ft_consumo_mensal = (
        df_leituras_gd.withColumn(
            'consumo_kwh_analitico',
            F.coalesce(F.col('volume_analise_kwh'), mediana_fase, mediana_global).cast(
                DecimalType(18, 2)
            ),
        )
        .withColumn(
            'metodo_consumo',
            F.when(F.col('volume_analise_kwh').isNotNull(), F.lit('LEITURA_UTILIZAVEL'))
            .when(mediana_fase.isNotNull(), F.lit('MEDIANA_FASE_UC'))
            .when(mediana_global.isNotNull(), F.lit('MEDIANA_HISTORICA_UC'))
            .otherwise(F.lit('INDISPONIVEL')),
        )
        .withColumn('versao_regras_gold', F.lit(VERSAO_REGRAS))
        .select(
            'gd_uc_sk',
            'id_uc',
            'mes_referencia',
            'competencia',
            'municipio',
            'bairro',
            'classe',
            'subgrupo_tarifario',
            'alimentador',
            'modalidade',
            'faixa_potencia',
            'data_conexao',
            'competencia_conexao',
            'flag_gd_ativa_competencia',
            'fase_gd',
            'potencia_kwp',
            'energia_injetada_kwh_media_mensal',
            'consumo_kwh_silver',
            'volume_analise_kwh',
            'consumo_kwh_analitico',
            'situacao_volume',
            'metodo_consumo',
            'tipo_leitura',
            'flag_sinal_corrigido',
            'flag_troca_medidor',
            'flag_sem_leitura_anterior',
            'flag_sinal_nao_confirmado',
            'versao_regras_gold',
        )
    )

    # 4. Agregação antes vs depois da conexão por UC (_agg) para enriquecer a dimensão de impacto
    df_impacto_uc_agg = df_ft_consumo_mensal.groupBy('id_uc').agg(
        F.count('*').alias('qtd_competencias_total'),
        F.count(F.when(~F.col('flag_gd_ativa_competencia'), 1)).alias(
            'qtd_competencias_antes_conexao'
        ),
        F.count(F.when(F.col('flag_gd_ativa_competencia'), 1)).alias(
            'qtd_competencias_apos_conexao'
        ),
        F.count(
            F.when(~F.col('flag_gd_ativa_competencia') & F.col('volume_analise_kwh').isNotNull(), 1)
        ).alias('qtd_leituras_utilizaveis_antes'),
        F.count(
            F.when(F.col('flag_gd_ativa_competencia') & F.col('volume_analise_kwh').isNotNull(), 1)
        ).alias('qtd_leituras_utilizaveis_apos'),
        F.round(
            F.avg(F.when(~F.col('flag_gd_ativa_competencia'), F.col('volume_analise_kwh'))),
            2,
        )
        .cast(DecimalType(18, 2))
        .alias('consumo_medio_antes_kwh'),
        F.round(
            F.avg(F.when(F.col('flag_gd_ativa_competencia'), F.col('volume_analise_kwh'))),
            2,
        )
        .cast(DecimalType(18, 2))
        .alias('consumo_medio_apos_kwh'),
        F.percentile_approx(
            F.when(~F.col('flag_gd_ativa_competencia'), F.col('volume_analise_kwh')),
            0.5,
        )
        .cast(DecimalType(18, 2))
        .alias('consumo_mediano_antes_kwh'),
        F.percentile_approx(
            F.when(F.col('flag_gd_ativa_competencia'), F.col('volume_analise_kwh')),
            0.5,
        )
        .cast(DecimalType(18, 2))
        .alias('consumo_mediano_apos_kwh'),
        _soma_decimal(
            F.when(
                F.col('competencia').between(
                    F.to_date(F.lit('2022-07-01')), F.to_date(F.lit('2023-06-01'))
                ),
                F.col('volume_analise_kwh'),
            )
        ).alias('volume_p0_utilizavel_kwh'),
        _soma_decimal(
            F.when(
                F.col('competencia').between(
                    F.to_date(F.lit('2025-07-01')), F.to_date(F.lit('2026-06-01'))
                ),
                F.col('volume_analise_kwh'),
            )
        ).alias('volume_p1_utilizavel_kwh'),
        _soma_decimal(
            F.when(
                F.col('competencia').between(
                    F.to_date(F.lit('2022-07-01')), F.to_date(F.lit('2023-06-01'))
                ),
                F.col('consumo_kwh_analitico'),
            )
        ).alias('volume_p0_analitico_kwh'),
        _soma_decimal(
            F.when(
                F.col('competencia').between(
                    F.to_date(F.lit('2025-07-01')), F.to_date(F.lit('2026-06-01'))
                ),
                F.col('consumo_kwh_analitico'),
            )
        ).alias('volume_p1_analitico_kwh'),
    )

    tem_base_antes_apos = (
        F.col('consumo_medio_antes_kwh').isNotNull()
        & (F.col('consumo_medio_antes_kwh') > 0)
        & F.col('consumo_medio_apos_kwh').isNotNull()
    )
    queda_media_kwh = F.when(
        tem_base_antes_apos,
        (F.col('consumo_medio_antes_kwh') - F.col('consumo_medio_apos_kwh')).cast(
            DecimalType(18, 2)
        ),
    )
    queda_media_pct = F.when(
        tem_base_antes_apos,
        F.round(
            (F.col('consumo_medio_antes_kwh') - F.col('consumo_medio_apos_kwh'))
            * F.lit(100)
            / F.col('consumo_medio_antes_kwh'),
            2,
        ).cast(DecimalType(10, 2)),
    )

    df_tb_gd_uc = (
        df_gd_base.join(df_impacto_uc_agg, 'id_uc', 'left')
        .fillna(
            0,
            subset=[
                'qtd_competencias_total',
                'qtd_competencias_antes_conexao',
                'qtd_competencias_apos_conexao',
                'qtd_leituras_utilizaveis_antes',
                'qtd_leituras_utilizaveis_apos',
            ],
        )
        .withColumn(
            'volume_p0_utilizavel_kwh',
            F.coalesce(F.col('volume_p0_utilizavel_kwh'), F.lit(0)).cast(DecimalType(28, 2)),
        )
        .withColumn(
            'volume_p1_utilizavel_kwh',
            F.coalesce(F.col('volume_p1_utilizavel_kwh'), F.lit(0)).cast(DecimalType(28, 2)),
        )
        .withColumn(
            'volume_p0_analitico_kwh',
            F.coalesce(F.col('volume_p0_analitico_kwh'), F.lit(0)).cast(DecimalType(28, 2)),
        )
        .withColumn(
            'volume_p1_analitico_kwh',
            F.coalesce(F.col('volume_p1_analitico_kwh'), F.lit(0)).cast(DecimalType(28, 2)),
        )
        .withColumn('queda_consumo_media_kwh', queda_media_kwh)
        .withColumn('queda_consumo_pct', queda_media_pct)
        .withColumn(
            'flag_queda_acima_30_pct',
            F.when(tem_base_antes_apos, F.col('queda_consumo_pct') > F.lit(30)),
        )
        .withColumn(
            'razao_queda_sobre_injecao',
            F.when(
                tem_base_antes_apos & (F.col('energia_injetada_kwh_media_mensal') > 0),
                F.round(
                    F.col('queda_consumo_media_kwh') / F.col('energia_injetada_kwh_media_mensal'),
                    4,
                ).cast(DecimalType(10, 4)),
            ),
        )
        .withColumn('versao_regras_gold', F.lit(VERSAO_REGRAS))
        .select(
            'gd_uc_sk',
            'id_uc',
            'municipio',
            'bairro',
            'classe',
            'subgrupo_tarifario',
            'alimentador',
            'status',
            'data_ligacao',
            'data_conexao',
            'competencia_conexao',
            'modalidade',
            'modalidade_origem',
            'faixa_potencia',
            'potencia_kwp',
            'energia_injetada_kwh_media_mensal',
            'kwh_por_kwp_mes',
            'fl_correcao_escala',
            'fl_data_inconsistente',
            'fl_deduplicado',
            'qtd_competencias_total',
            'qtd_competencias_antes_conexao',
            'qtd_competencias_apos_conexao',
            'qtd_leituras_utilizaveis_antes',
            'qtd_leituras_utilizaveis_apos',
            'consumo_medio_antes_kwh',
            'consumo_medio_apos_kwh',
            'consumo_mediano_antes_kwh',
            'consumo_mediano_apos_kwh',
            'queda_consumo_media_kwh',
            'queda_consumo_pct',
            'flag_queda_acima_30_pct',
            'razao_queda_sobre_injecao',
            'volume_p0_utilizavel_kwh',
            'volume_p1_utilizavel_kwh',
            'volume_p0_analitico_kwh',
            'volume_p1_analitico_kwh',
            'versao_regras_gold',
        )
        .orderBy('id_uc')
    )

    # 5. Resumo mensal: contagens e potências vêm do cadastro de GD cruzado com o
    # calendário mensal completo; volumes continuam nas leituras. Assim, uma UC sem
    # leitura no mês entra na contagem e na potência, antes ou depois da conexão.
    chaves_resumo = ['competencia', 'municipio', 'bairro', 'classe', 'modalidade']
    limites = df_gd_base.agg(
        F.min('competencia_conexao').alias('min_conexao'),
        F.max('competencia_conexao').alias('max_conexao'),
    ).crossJoin(
        df_ft_consumo_mensal.agg(
            F.min('competencia').alias('min_leitura'),
            F.max('competencia').alias('max_leitura'),
        )
    )
    df_calendario = limites.select(
        F.explode(
            F.sequence(
                F.least(F.col('min_conexao'), F.col('min_leitura')),
                F.greatest(F.col('max_conexao'), F.col('max_leitura')),
                F.expr('INTERVAL 1 MONTH'),
            )
        ).alias('competencia')
    )
    df_uc_grade = df_gd_base.select(
        'id_uc',
        'municipio',
        'bairro',
        'classe',
        'modalidade',
        'potencia_kwp',
        'energia_injetada_kwh_media_mensal',
        'competencia_conexao',
    )
    ativa = F.col('competencia') >= F.col('competencia_conexao')
    df_estoque = (
        df_calendario.crossJoin(df_uc_grade)
        .groupBy(*chaves_resumo)
        .agg(
            F.count('*').alias('quantidade_ucs_gd_cadastradas'),
            F.count(F.when(ativa, 1)).alias('quantidade_ucs_gd_ativas'),
            F.count(F.when(F.col('competencia') == F.col('competencia_conexao'), 1)).alias(
                'quantidade_novas_conexoes_mes'
            ),
            _soma_decimal('potencia_kwp').alias('potencia_cadastrada_kwp'),
            _soma_decimal(F.when(ativa, F.col('potencia_kwp'))).alias(
                'potencia_instalada_ativa_kwp'
            ),
            _soma_decimal(F.when(ativa, F.col('energia_injetada_kwh_media_mensal'))).alias(
                'energia_injetada_estimada_ativa_kwh'
            ),
        )
    )
    df_volumes = df_ft_consumo_mensal.groupBy(*chaves_resumo).agg(
        F.count(F.when(F.col('volume_analise_kwh').isNotNull(), 1)).alias(
            'quantidade_leituras_utilizaveis'
        ),
        F.count(F.when(F.col('volume_analise_kwh').isNull(), 1)).alias(
            'quantidade_leituras_bloqueadas'
        ),
        _soma_decimal('volume_analise_kwh').alias('volume_utilizavel_kwh'),
        _soma_decimal(
            F.when(F.col('flag_gd_ativa_competencia'), F.col('volume_analise_kwh'))
        ).alias('volume_utilizavel_gd_ativa_kwh'),
        _soma_decimal(
            F.when(~F.col('flag_gd_ativa_competencia'), F.col('volume_analise_kwh'))
        ).alias('volume_utilizavel_antes_conexao_kwh'),
        _soma_decimal('consumo_kwh_analitico').alias('volume_analitico_kwh'),
        _soma_decimal(
            F.when(F.col('flag_gd_ativa_competencia'), F.col('consumo_kwh_analitico'))
        ).alias('volume_analitico_gd_ativa_kwh'),
        _soma_decimal(
            F.when(~F.col('flag_gd_ativa_competencia'), F.col('consumo_kwh_analitico'))
        ).alias('volume_analitico_antes_conexao_kwh'),
    )
    zero_dec = F.lit(0).cast(DecimalType(28, 2))
    df_resumo_mensal = (
        df_estoque.join(df_volumes, chaves_resumo, 'full')
        .select(
            *chaves_resumo,
            F.coalesce(F.col('quantidade_ucs_gd_cadastradas'), F.lit(0)).alias(
                'quantidade_ucs_gd_cadastradas'
            ),
            F.coalesce(F.col('quantidade_ucs_gd_ativas'), F.lit(0)).alias(
                'quantidade_ucs_gd_ativas'
            ),
            F.coalesce(F.col('quantidade_novas_conexoes_mes'), F.lit(0)).alias(
                'quantidade_novas_conexoes_mes'
            ),
            F.coalesce(F.col('potencia_cadastrada_kwp'), zero_dec).alias('potencia_cadastrada_kwp'),
            F.coalesce(F.col('potencia_instalada_ativa_kwp'), zero_dec).alias(
                'potencia_instalada_ativa_kwp'
            ),
            F.coalesce(F.col('energia_injetada_estimada_ativa_kwh'), zero_dec).alias(
                'energia_injetada_estimada_ativa_kwh'
            ),
            F.coalesce(F.col('quantidade_leituras_utilizaveis'), F.lit(0)).alias(
                'quantidade_leituras_utilizaveis'
            ),
            F.coalesce(F.col('quantidade_leituras_bloqueadas'), F.lit(0)).alias(
                'quantidade_leituras_bloqueadas'
            ),
            F.coalesce(F.col('volume_utilizavel_kwh'), zero_dec).alias('volume_utilizavel_kwh'),
            F.coalesce(F.col('volume_utilizavel_gd_ativa_kwh'), zero_dec).alias(
                'volume_utilizavel_gd_ativa_kwh'
            ),
            F.coalesce(F.col('volume_utilizavel_antes_conexao_kwh'), zero_dec).alias(
                'volume_utilizavel_antes_conexao_kwh'
            ),
            F.coalesce(F.col('volume_analitico_kwh'), zero_dec).alias('volume_analitico_kwh'),
            F.coalesce(F.col('volume_analitico_gd_ativa_kwh'), zero_dec).alias(
                'volume_analitico_gd_ativa_kwh'
            ),
            F.coalesce(F.col('volume_analitico_antes_conexao_kwh'), zero_dec).alias(
                'volume_analitico_antes_conexao_kwh'
            ),
        )
        .withColumn('versao_regras_gold', F.lit(VERSAO_REGRAS))
        .orderBy(*chaves_resumo)
    )

    return {
        'gd_uc': df_gd_uc,
        'tb_gd_uc': df_tb_gd_uc,
        'tb_gd_consumo_mensal': df_ft_consumo_mensal,
        'tb_gd_resumo_mensal': df_resumo_mensal,
    }


def validar_reconciliacao(gd_silver: DataFrame, tabelas: dict[str, DataFrame]) -> None:
    """Confere UCs, potência, volumes e o fechamento das conexões com o cadastro."""
    gd_uc = tabelas['gd_uc']
    tb_gd_uc = tabelas['tb_gd_uc']
    mensal_uc = tabelas['tb_gd_consumo_mensal']
    resumo = tabelas['tb_gd_resumo_mensal']

    _exigir_chave(gd_uc, ['id_uc'], 'gd_uc')
    _exigir_chave(tb_gd_uc, ['id_uc'], 'tb_gd_uc')
    _exigir_chave(tb_gd_uc, ['gd_uc_sk'], 'tb_gd_uc')
    _exigir_chave(mensal_uc, ['id_uc', 'competencia'], 'tb_gd_consumo_mensal')
    _exigir_chave(
        resumo,
        ['competencia', 'municipio', 'bairro', 'classe', 'modalidade'],
        'tb_gd_resumo_mensal',
    )

    # 1. Reconciliação exata entre gd_silver, gd_uc e tb_gd_uc
    esperado_contrato = gd_silver.select(
        F.col('id_uc'),
        F.col('potencia_kwp').cast(DecimalType(18, 2)).alias('potencia_kw'),
        F.col('data_conexao'),
    )
    for a, b in [(esperado_contrato, gd_uc), (gd_uc, esperado_contrato)]:
        if a.exceptAll(b).limit(1).count():
            raise ValueError('gd_uc diverge da fonte Silver de GD')

    esperado_dim = gd_silver.select(
        'id_uc',
        'potencia_kwp',
        'energia_injetada_kwh_media_mensal',
        'data_conexao',
        'modalidade',
    )
    obtido_dim = tb_gd_uc.select(
        'id_uc',
        'potencia_kwp',
        'energia_injetada_kwh_media_mensal',
        'data_conexao',
        'modalidade',
    )
    for a, b in [(esperado_dim, obtido_dim), (obtido_dim, esperado_dim)]:
        if a.exceptAll(b).limit(1).count():
            raise ValueError('tb_gd_uc diverge da fonte Silver de GD')

    # 2. Volumes e contagens de leitura reconciliam com o fato mensal.
    # Meses sem leitura ficam zerados e não derrubam conexão nem potência.
    chaves_resumo = ['competencia', 'municipio', 'bairro', 'classe', 'modalidade']
    esperado_volumes = mensal_uc.groupBy(*chaves_resumo).agg(
        F.count(F.when(F.col('volume_analise_kwh').isNotNull(), 1)).alias(
            'quantidade_leituras_utilizaveis'
        ),
        F.count(F.when(F.col('volume_analise_kwh').isNull(), 1)).alias(
            'quantidade_leituras_bloqueadas'
        ),
        _soma_decimal('volume_analise_kwh').alias('volume_utilizavel_kwh'),
        _soma_decimal(
            F.when(F.col('flag_gd_ativa_competencia'), F.col('volume_analise_kwh'))
        ).alias('volume_utilizavel_gd_ativa_kwh'),
        _soma_decimal(
            F.when(~F.col('flag_gd_ativa_competencia'), F.col('volume_analise_kwh'))
        ).alias('volume_utilizavel_antes_conexao_kwh'),
        _soma_decimal('consumo_kwh_analitico').alias('volume_analitico_kwh'),
        _soma_decimal(
            F.when(F.col('flag_gd_ativa_competencia'), F.col('consumo_kwh_analitico'))
        ).alias('volume_analitico_gd_ativa_kwh'),
        _soma_decimal(
            F.when(~F.col('flag_gd_ativa_competencia'), F.col('consumo_kwh_analitico'))
        ).alias('volume_analitico_antes_conexao_kwh'),
    )
    obtido_volumes = resumo.select(*esperado_volumes.columns)
    juncao = esperado_volumes.alias('esperado').join(
        obtido_volumes.alias('obtido'), chaves_resumo, 'full'
    )

    def _diverge(coluna, tipo):
        esquerda = F.coalesce(F.col(f'esperado.{coluna}').cast(tipo), F.lit(0).cast(tipo))
        direita = F.coalesce(F.col(f'obtido.{coluna}').cast(tipo), F.lit(0).cast(tipo))
        return esquerda != direita

    condicao = F.lit(False)
    for coluna in (
        'quantidade_leituras_utilizaveis',
        'quantidade_leituras_bloqueadas',
    ):
        condicao = condicao | _diverge(coluna, LongType())
    for coluna in (
        'volume_utilizavel_kwh',
        'volume_utilizavel_gd_ativa_kwh',
        'volume_utilizavel_antes_conexao_kwh',
        'volume_analitico_kwh',
        'volume_analitico_gd_ativa_kwh',
        'volume_analitico_antes_conexao_kwh',
    ):
        condicao = condicao | _diverge(coluna, DecimalType(28, 2))
    if juncao.filter(condicao).limit(1).count():
        raise ValueError('tb_gd_resumo_mensal diverge do detalhe mensal nas leituras ou volumes')

    # 3. Conexões e estoque ativo fecham com o cadastro, independente de leitura.
    n_ucs = gd_silver.count()
    total_novas = resumo.agg(F.sum('quantidade_novas_conexoes_mes').alias('novas')).first()['novas']
    if total_novas != n_ucs:
        raise ValueError(f'tb_gd_resumo_mensal: novas conexões {total_novas} != UCs de GD {n_ucs}')
    por_mes = resumo.groupBy('competencia').agg(
        F.sum('quantidade_novas_conexoes_mes').alias('novas'),
        F.sum('quantidade_ucs_gd_ativas').alias('ativas'),
    )
    janela_mes = Window.orderBy('competencia').rowsBetween(
        Window.unboundedPreceding, Window.currentRow
    )
    estoque_divergente = (
        por_mes.withColumn('estoque', F.sum('novas').over(janela_mes))
        .filter(F.col('estoque') != F.col('ativas'))
        .limit(1)
        .count()
    )
    if estoque_divergente:
        raise ValueError('tb_gd_resumo_mensal: estoque ativo não acompanha as novas conexões')
    ultima = resumo.agg(F.max('competencia').alias('competencia')).first()['competencia']
    ativas_ultima = (
        resumo.filter(F.col('competencia') == F.lit(ultima))
        .agg(F.sum('quantidade_ucs_gd_ativas').alias('ativas'))
        .first()['ativas']
    )
    if ativas_ultima != n_ucs:
        raise ValueError(
            'tb_gd_resumo_mensal: UCs ativas na última competência '
            f'{ativas_ultima} != UCs de GD {n_ucs}'
        )
    potencia_ultima = (
        resumo.filter(F.col('competencia') == F.lit(ultima))
        .agg(_soma_decimal('potencia_instalada_ativa_kwp').alias('potencia'))
        .first()['potencia']
    )
    potencia_cadastro = gd_silver.agg(
        F.coalesce(F.sum('potencia_kwp'), F.lit(0)).cast(DecimalType(28, 2)).alias('potencia')
    ).first()['potencia']
    if potencia_ultima != potencia_cadastro:
        raise ValueError(
            'tb_gd_resumo_mensal: potência ativa na última competência '
            f'{potencia_ultima} diverge do cadastro {potencia_cadastro}'
        )
    # Toda competência tem a coorte inteira de GD, conectada ou não, com ou sem leitura
    coorte_divergente = (
        resumo.groupBy('competencia')
        .agg(
            F.sum('quantidade_ucs_gd_cadastradas').alias('ucs'),
            _soma_decimal('potencia_cadastrada_kwp').alias('potencia'),
        )
        .filter((F.col('ucs') != n_ucs) | (F.col('potencia') != F.lit(potencia_cadastro)))
        .limit(1)
        .count()
    )
    if coorte_divergente:
        raise ValueError(
            'tb_gd_resumo_mensal: UCs ou potência cadastradas diferem do cadastro de GD '
            'em alguma competência'
        )

    # 4. Regras de integridade algébrica do resumo mensal
    _validar_regras(
        resumo,
        {
            'volume_utilizavel_nao_reconciliado': (
                F.col('volume_utilizavel_kwh')
                != F.col('volume_utilizavel_gd_ativa_kwh')
                + F.col('volume_utilizavel_antes_conexao_kwh')
            ),
            'volume_analitico_nao_reconciliado': (
                F.col('volume_analitico_kwh')
                != F.col('volume_analitico_gd_ativa_kwh')
                + F.col('volume_analitico_antes_conexao_kwh')
            ),
            'leituras_excedem_coorte': (
                F.col('quantidade_leituras_utilizaveis') + F.col('quantidade_leituras_bloqueadas')
                > F.col('quantidade_ucs_gd_cadastradas')
            ),
            'novas_maior_que_ativas': (
                F.col('quantidade_novas_conexoes_mes') > F.col('quantidade_ucs_gd_ativas')
            ),
            'ativas_maior_que_cadastradas': (
                F.col('quantidade_ucs_gd_ativas') > F.col('quantidade_ucs_gd_cadastradas')
            ),
        },
        'tb_gd_resumo_mensal',
    )


def publicar_delta(
    spark, tabelas: dict[str, DataFrame], destinos: dict[str, str], validar
) -> tuple:
    """Publica e confere as tabelas Gold; recupera os destinos afetados em caso de falha."""
    anteriores = {}
    for nome in tabelas:
        destino = destinos[nome]
        if spark.catalog.tableExists(destino):
            detalhe = spark.sql(f'DESCRIBE DETAIL {destino}').first()
            if detalhe['format'] != 'delta':
                raise ValueError(f'O destino precisa ser Delta: {destino}')
            anteriores[nome] = spark.sql(f'DESCRIBE HISTORY {destino} LIMIT 1').first()['version']
        else:
            anteriores[nome] = None

    afetadas = []
    persistidas, versoes = {}, {}
    try:
        for nome, df in tabelas.items():
            destino = destinos[nome]
            afetadas.append(nome)
            (
                df.write.format('delta')
                .mode('overwrite')
                .option('overwriteSchema', 'true')
                .saveAsTable(destino)
            )
            versao = spark.sql(f'DESCRIBE HISTORY {destino} LIMIT 1').first()['version']
            persistido = spark.read.option('versionAsOf', versao).table(destino)
            esperado = [(c.name, c.dataType.simpleString()) for c in df.schema]
            gravado = [(c.name, c.dataType.simpleString()) for c in persistido.schema]
            if esperado != gravado:
                raise ValueError(f'Schema persistido divergente: {destino}')
            if (
                df.exceptAll(persistido).limit(1).count()
                or persistido.exceptAll(df).limit(1).count()
            ):
                raise ValueError(f'Conteúdo persistido divergente: {destino}')
            persistidas[nome] = persistido
            versoes[nome] = {'tabela': destino, 'versao_delta': versao}
        validar(persistidas)
    except Exception as erro:
        falhas = []
        for nome in reversed(afetadas):
            destino = destinos[nome]
            try:
                versao = anteriores[nome]
                if versao is None:
                    spark.sql(f'DROP TABLE IF EXISTS {destino}')
                else:
                    spark.sql(f'RESTORE TABLE {destino} TO VERSION AS OF {versao}')
            except Exception as falha:
                falhas.append(RuntimeError(f'Falha ao recuperar {destino}: {falha}'))
        if falhas:
            raise ExceptionGroup(
                'Publicação falhou e a recuperação ficou incompleta', [erro, *falhas]
            ) from erro
        raise
    return persistidas, versoes

In [0]:
# ruff: noqa: F821
import re


parametros = {
    'tabela_gd': 'v-energy.silver.tb_geracao_distribuida',
    'tabela_cadastro': 'v-energy.silver.tb_unidades_consumidoras',
    'tabela_leituras': 'v-energy.silver.tb_leituras',
    'schema_gold': 'v-energy.gold',
    'modo': 'validar',
}
for nome, padrao in parametros.items():
    dbutils.widgets.text(nome, padrao)
config = {nome: dbutils.widgets.get(nome).strip() for nome in parametros}
if config['modo'] not in {'validar', 'publicar'}:
    raise ValueError('modo deve ser validar ou publicar')


def qualificar(nome, partes=3):
    segmentos = nome.split('.')
    if len(segmentos) != partes or any(
        not re.fullmatch(r'[A-Za-z0-9_-]+', segmento) for segmento in segmentos
    ):
        raise ValueError(f'Identificador inválido: {nome}')
    return '.'.join(f'`{segmento}`' for segmento in segmentos)


schema_gold = qualificar(config['schema_gold'], 2)
if config['schema_gold'].split('.')[1] in {'silver', 'bronze', 'default'}:
    raise ValueError('O destino deve ser um schema próprio da Gold')
fontes = {}
print(f'Schema Gold: {schema_gold} | Modo: {config["modo"]}')
if config['modo'] == 'publicar':
    print(f'ATENÇÃO: modo=publicar. As tabelas Gold serão gravadas em {schema_gold}')

Schema Gold: `v-energy`.`gold` | Modo: validar


## 1. Leitura consistente da Silver (`VERSION AS OF`)

Fixa a versão Delta atual de cada uma das três tabelas Silver no início da execução (`tabela_gd`, `tabela_cadastro` e `tabela_leituras`), garantindo que validações, transformações e escrita leiam exatamente o mesmo snapshot. Se os nomes físicos do cadastro ou das leituras estiverem em outro catálogo/schema no workspace (ex.: `workspace.default.silver_unidades_consumidoras` ou `venergy.silver.leituras`), informe-os nos widgets acima.

In [0]:
# ruff: noqa: F821
def ler_snapshot(chave):
    tabela = qualificar(config[chave])
    if not spark.catalog.tableExists(tabela):
        raise ValueError(
            f'Tabela Silver não encontrada: {tabela}. '
            f'Publique a Silver correspondente ou ajuste o widget {chave}.'
        )
    detalhe = spark.sql(f'DESCRIBE DETAIL {tabela}').first()
    if detalhe['format'] != 'delta':
        raise ValueError(f'A fonte precisa ser Delta: {tabela}')
    historico = spark.sql(f'DESCRIBE HISTORY {tabela} LIMIT 1').first()
    fontes[chave] = {
        'tabela': config[chave],
        'id_delta': detalhe['id'],
        'versao_delta': historico['version'],
        'timestamp_commit': str(historico['timestamp']),
    }
    print(f'{chave}: {tabela} (versão {historico["version"]})')
    return spark.read.option('versionAsOf', historico['version']).table(tabela)


df_gd_silver = ler_snapshot('tabela_gd')
df_cadastro_silver = ler_snapshot('tabela_cadastro')
df_leituras_silver = ler_snapshot('tabela_leituras')

tabelas = construir_gold_gd(df_gd_silver, df_cadastro_silver, df_leituras_silver)
validar_reconciliacao(df_gd_silver, tabelas)
for nome, df_tab in tabelas.items():
    print(f'[OK] {nome}: {df_tab.count():,} registros validados em memória')

total_novas = (
    tabelas['tb_gd_resumo_mensal']
    .agg(F.sum('quantidade_novas_conexoes_mes').alias('novas'))
    .first()['novas']
)
total_ucs = tabelas['gd_uc'].count()
print(f'[OK] novas conexões no resumo: {total_novas:,} (cadastro de GD: {total_ucs:,})')

tabela_gd: `v-energy`.`silver`.`tb_geracao_distribuida` (versão 16)
tabela_cadastro: `v-energy`.`silver`.`tb_unidades_consumidoras` (versão 41)
tabela_leituras: `v-energy`.`silver`.`tb_leituras` (versão 4)


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


[OK] gd_uc: 3,510 registros validados em memória
[OK] tb_gd_uc: 3,510 registros validados em memória
[OK] tb_gd_consumo_mensal: 168,083 registros validados em memória
[OK] tb_gd_resumo_mensal: 32,304 registros validados em memória
[OK] novas conexões no resumo: 3,510 (cadastro de GD: 3,510)


## 2. Verificação das perguntas de negócio do case

Confere diretamente nas tabelas Gold geradas:
- **Pergunta da Diretora de Tecnologia (Patrícia Lins):** quantas UCs com painel solar em Olinda tiveram queda de consumo acima de 30% após a conexão?
- **Pergunta do Diretor Comercial (Henrique Duarte):** qual foi a variação de volume das UCs com GD entre o primeiro ano (`P0`: jul/2022–jun/2023) e os últimos 12 meses (`P1`: jul/2025–jun/2026) por classe e município?
- **Evolução mensal:** estoque ativo de kWp e consumo antes × após a conexão.

In [0]:
# ruff: noqa: F821
tb_gd_uc = tabelas['tb_gd_uc']
tb_resumo = tabelas['tb_gd_resumo_mensal']

print('=== 1) UCs com GD em Olinda e queda > 30% após a conexão ===')
display(
    tb_gd_uc.filter(F.upper(F.col('municipio')) == F.lit('OLINDA')).agg(
        F.count('*').alias('total_ucs_gd_olinda'),
        F.count(F.when(F.col('flag_queda_acima_30_pct').isNotNull(), 1)).alias(
            'ucs_com_base_antes_e_apos'
        ),
        F.count(F.when(F.col('flag_queda_acima_30_pct'), 1)).alias('ucs_queda_acima_30_pct'),
    )
)

print('=== 2) Impacto de volume P0 (jul/22-jun/23) vs P1 (jul/25-jun/26) por classe ===')
display(
    tb_gd_uc.groupBy('classe')
    .agg(
        F.count('*').alias('qtd_ucs_gd'),
        F.sum('potencia_kwp').alias('potencia_total_kwp'),
        F.sum('volume_p0_utilizavel_kwh').alias('volume_p0_utilizavel_kwh'),
        F.sum('volume_p1_utilizavel_kwh').alias('volume_p1_utilizavel_kwh'),
        F.round(
            (F.sum('volume_p1_utilizavel_kwh') - F.sum('volume_p0_utilizavel_kwh'))
            * 100
            / F.sum('volume_p0_utilizavel_kwh'),
            2,
        ).alias('variacao_utilizavel_pct'),
        F.sum('volume_p0_analitico_kwh').alias('volume_p0_analitico_kwh'),
        F.sum('volume_p1_analitico_kwh').alias('volume_p1_analitico_kwh'),
        F.round(
            (F.sum('volume_p1_analitico_kwh') - F.sum('volume_p0_analitico_kwh'))
            * 100
            / F.sum('volume_p0_analitico_kwh'),
            2,
        ).alias('variacao_analitico_pct'),
    )
    .orderBy('classe')
)

print('=== 3) Evolução mensal do estoque ativo de GD ===')
display(
    tb_resumo.groupBy('competencia')
    .agg(
        F.sum('quantidade_novas_conexoes_mes').alias('novas_conexoes'),
        F.sum('quantidade_ucs_gd_ativas').alias('ucs_gd_ativas'),
        F.sum('potencia_instalada_ativa_kwp').alias('potencia_ativa_kwp'),
        F.sum('volume_analitico_kwh').alias('volume_analitico_total_kwh'),
    )
    .orderBy('competencia')
)

=== 1) UCs com GD em Olinda e queda > 30% após a conexão ===


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


total_ucs_gd_olinda,ucs_com_base_antes_e_apos,ucs_queda_acima_30_pct
328,327,327


=== 2) Impacto de volume P0 (jul/22-jun/23) vs P1 (jul/25-jun/26) por classe ===


classe,qtd_ucs_gd,potencia_total_kwp,volume_p0_utilizavel_kwh,volume_p1_utilizavel_kwh,variacao_utilizavel_pct,volume_p0_analitico_kwh,volume_p1_analitico_kwh,variacao_analitico_pct
Comercial,383,13168.98,7712547.00,2974053.00,-61.44,7713843.00,3012778.00,-60.94
Residencial,3127,18696.62,10582699.00,4032239.00,-61.90,10584385.00,4079920.00,-61.45


=== 3) Evolução mensal do estoque ativo de GD ===


competencia,novas_conexoes,ucs_gd_ativas,potencia_ativa_kwp,volume_analitico_total_kwh
2022-07-01,0,0,0.00,958558.00
2022-08-01,0,0,0.00,960735.00
2022-09-01,0,0,0.00,1124911.00
2022-10-01,0,0,0.00,1458167.00
2022-11-01,0,0,0.00,1754285.00
2022-12-01,0,0,0.00,1870161.00
2023-01-01,0,0,0.00,1969959.00
2023-02-01,0,0,0.00,1895419.00
2023-03-01,0,0,0.00,1912156.00
2023-04-01,0,0,0.00,1767599.00


## 3. Publicação Delta com `RESTORE`

A publicação só ocorre quando `modo=publicar`. Antes da primeira escrita, registra a versão atual de cada tabela de destino e, se a escrita ou a conferência pós-escrita falhar, restaura os destinos afetados com `RESTORE TABLE ... TO VERSION AS OF` (ou `DROP TABLE IF EXISTS` na primeira carga).

In [0]:
# ruff: noqa: F821
if config['modo'] == 'validar':
    print('Validação concluída. Nenhum schema ou tabela foi gravado.')
else:
    if any(dbutils.widgets.get(nome).strip() != valor for nome, valor in config.items()):
        raise ValueError('Parâmetros mudaram durante a execução. Execute novamente desde o início.')
    destinos = {nome: f'{schema_gold}.`{nome}`' for nome in tabelas}
    for chave, fonte in fontes.items():
        atual = spark.sql(f'DESCRIBE DETAIL {qualificar(config[chave])}').first()
        if atual['id'] != fonte['id_delta']:
            raise ValueError(f'A identidade da tabela fonte mudou: {chave}')
    if set(destinos.values()) & {qualificar(config[c]) for c in fontes}:
        raise ValueError('Uma tabela de destino coincide com uma fonte')

    spark.sql(f'CREATE SCHEMA IF NOT EXISTS {schema_gold}')
    persistidas, versoes_gold = publicar_delta(
        spark,
        tabelas,
        destinos,
        lambda gravadas: validar_reconciliacao(df_gd_silver, gravadas),
    )
    for nome, info in versoes_gold.items():
        print(f'Publicada: {info["tabela"]} (versão {info["versao_delta"]})')

Validação concluída. Nenhum schema ou tabela foi gravado.
